# ESA WorldCover

No account needed. 10m land cover tiles covering the Banten bbox (computed from `common.BANTEN_BBOX`).

In [ ]:
# make sure we're running from scripts/download so `import common` resolves
import os, sys

NOTEBOOK_DIR = os.path.abspath(".")
if os.path.basename(NOTEBOOK_DIR) != "download":
    NOTEBOOK_DIR = os.path.join(NOTEBOOK_DIR, "scripts", "download")
os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)
print("cwd:", os.getcwd())

In [ ]:
import math

from common import BANTEN_BBOX, RAW_DIR, download_file, ensure_dir, get_logger

log = get_logger("worldcover")

VERSION = "v200"
YEAR = 2021
BASE_URL = f"https://esa-worldcover.s3.eu-central-1.amazonaws.com/{VERSION}/{YEAR}/map"
OUT_DIR = RAW_DIR / "worldcover"

GRID_STEP = 3  # WorldCover tiles are 3x3 degrees


def tile_name(lat0: int, lon0: int) -> str:
    ns = f"S{abs(lat0):02d}" if lat0 < 0 else f"N{lat0:02d}"
    ew = f"W{abs(lon0):03d}" if lon0 < 0 else f"E{lon0:03d}"
    return f"{ns}{ew}"


def tiles_for_bbox(west: float, south: float, east: float, north: float, step: int = GRID_STEP) -> list[str]:
    lat0 = math.floor(south / step) * step
    lat1 = math.floor(north / step) * step
    lon0 = math.floor(west / step) * step
    lon1 = math.floor(east / step) * step

    names = []
    for lat in range(lat0, lat1 + 1, step):
        for lon in range(lon0, lon1 + 1, step):
            names.append(tile_name(lat, lon))
    return names


def main() -> None:
    ensure_dir(OUT_DIR)
    tiles = tiles_for_bbox(**BANTEN_BBOX)
    log.info("Tiles needed: %s", tiles)

    for tile in tiles:
        fname = f"ESA_WorldCover_10m_{YEAR}_{VERSION}_{tile}_Map.tif"
        url = f"{BASE_URL}/{fname}"
        dest = OUT_DIR / fname
        try:
            download_file(url, dest)
        except Exception as e:  # noqa: BLE001
            log.warning("Failed to download tile %s (%s): %s", tile, url, e)

    log.info("Done. Files are in %s", OUT_DIR)

In [ ]:
main()